# YOLO11n aquarium inference

Download the v1.0.0 model, verify SHA256, and run detection on one image.


In [ ]:
%pip install -q "ultralytics==8.4.115" "requests>=2.31,<3.0" "Pillow>=10.0,<12.0"

In [ ]:
import hashlib
import os
import re
from pathlib import Path
from urllib.parse import urlparse

import requests
from IPython.display import display
from PIL import Image
from ultralytics import YOLO

RELEASE_ASSET_URL = "https://github.com/CpfPatrick/aquarium-yolo11-portfolio/releases/download/v1.0.0/aquarium-yolo11n-imgsz960-best.pt"
EXPECTED_SHA256 = "ff7284669dd3d1b30b83f2d9eaacbdefdf8b50836f6f4e629bfd16f877a2f281"
MODEL_IMGSZ_RAW = "960"

MODEL_DIR = Path("models")
OUTPUT_DIR = Path("outputs")
MODEL_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def download_verified_model(url: str, expected_sha256: str) -> Path:
    if url.startswith("REPLACE_") or "<" in url or "PLACEHOLDER" in url:
        raise ValueError("Replace RELEASE_ASSET_URL with the GitHub Release asset URL.")

    expected = expected_sha256.strip().lower()
    if re.fullmatch(r"[0-9a-f]{64}", expected) is None:
        raise ValueError("EXPECTED_SHA256 must be the 64-character value from the sealed manifest.")

    filename = Path(urlparse(url).path).name
    if not filename.endswith(".pt"):
        raise ValueError("The Release asset URL must point to a .pt checkpoint.")

    destination = MODEL_DIR / filename
    temporary = destination.with_suffix(destination.suffix + ".part")

    if destination.exists() and sha256_file(destination) == expected:
        print(f"Using verified cached model: {destination}")
        return destination

    with requests.get(url, stream=True, timeout=60) as response:
        response.raise_for_status()
        with temporary.open("wb") as stream:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    stream.write(chunk)

    actual = sha256_file(temporary)
    if actual != expected:
        temporary.unlink(missing_ok=True)
        raise RuntimeError(f"Model SHA256 mismatch: expected {expected}, got {actual}.")

    temporary.replace(destination)
    print(f"Downloaded and verified: {destination}")
    print(f"SHA256: {actual}")
    return destination


MODEL_PATH = download_verified_model(RELEASE_ASSET_URL, EXPECTED_SHA256)
if not MODEL_IMGSZ_RAW.isdigit():
    raise ValueError("Replace MODEL_IMGSZ_RAW with the release manifest value.")
MODEL_IMGSZ = int(MODEL_IMGSZ_RAW)
if MODEL_IMGSZ not in {640, 960}:
    raise ValueError(f"Unexpected registered imgsz: {MODEL_IMGSZ}")

## Choose one image

In Colab, the next cell opens an upload dialog. In local Jupyter, set `LOCAL_IMAGE` to one image path.

In [ ]:
LOCAL_IMAGE = os.environ.get("AQUARIUM_IMAGE", "")  # Or set a local image path.

try:
    from google.colab import files  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if LOCAL_IMAGE:
    IMAGE_PATH = Path(LOCAL_IMAGE)
elif IN_COLAB:
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Upload exactly one image.")
    IMAGE_PATH = Path(next(iter(uploaded)))
else:
    if not LOCAL_IMAGE:
        raise ValueError("Set LOCAL_IMAGE to one image path before running this cell.")
    IMAGE_PATH = Path(LOCAL_IMAGE)

if not IMAGE_PATH.is_file():
    raise FileNotFoundError(IMAGE_PATH)

with Image.open(IMAGE_PATH) as source_image:
    source_image.verify()

print(f"Input image: {IMAGE_PATH}")

In [ ]:
model = YOLO(str(MODEL_PATH))
result = model.predict(
    source=str(IMAGE_PATH),
    imgsz=MODEL_IMGSZ,
    conf=0.25,
    iou=0.70,
    save=False,
    verbose=False,
)[0]

records = []
if result.boxes is not None:
    for xyxy, confidence, class_id in zip(
        result.boxes.xyxy.cpu().tolist(),
        result.boxes.conf.cpu().tolist(),
        result.boxes.cls.cpu().tolist(),
    ):
        class_id = int(class_id)
        records.append({
            "class_id": class_id,
            "class_name": result.names[class_id],
            "confidence": round(float(confidence), 6),
            "xyxy": [round(float(value), 2) for value in xyxy],
        })

annotated_bgr = result.plot()
annotated_rgb = annotated_bgr[..., ::-1]
annotated_image = Image.fromarray(annotated_rgb)
OUTPUT_IMAGE = OUTPUT_DIR / f"{IMAGE_PATH.stem}_prediction.jpg"
annotated_image.save(OUTPUT_IMAGE, quality=95)

display(annotated_image)
print(f"Detections: {len(records)}")
for record in records:
    print(record)
print(f"Saved annotated image: {OUTPUT_IMAGE.resolve()}")